In [1]:
import sqlite3
import glob
import os
import pandas as pd

# نسوي (أو نفتح) ملف قاعدة البيانات
conn = sqlite3.connect("../data/tourism.db")

# كل ملف CSV نظيف يصير جدول في قاعدة البيانات
for path in glob.glob("../data/clean/*.csv"):
    table_name = os.path.basename(path).replace(".csv", "")
    pd.read_csv(path).to_sql(table_name, conn, if_exists="replace", index=False)
    print("✅", table_name)

✅ inbound_origin
✅ inbound_purpose
✅ inbound_top10_countries
✅ kingdom_yearly
✅ monthly
✅ regions_yearly


In [2]:
# دالة صغيرة: نعطيها سؤال SQL، ترجّع جدول
def sql(query):
    return pd.read_sql(query, conn)

sql("""
    SELECT year, tourism_type, ROUND(value) AS tourists_000
    FROM kingdom_yearly
    WHERE indicator = 'Tourists Number (Overnight Visitors)'
      AND tourism_type IN ('Inbound', 'Domestic')
    ORDER BY tourism_type, year
""")

,year,tourism_type,tourists_000
0,2021,Domestic,63834.0
1,2022,Domestic,77837.0
2,2023,Domestic,81920.0
3,2024,Domestic,86157.0
4,2025,Domestic,93324.0
5,2021,Inbound,3477.0
6,2022,Inbound,16638.0
7,2023,Inbound,27424.0
8,2024,Inbound,29727.0
9,2025,Inbound,29255.0


In [3]:
sql("""
    WITH tourists AS (
        SELECT
            year,
            tourism_type,
            value,
            LAG(value) OVER (PARTITION BY tourism_type ORDER BY year) AS prev_value
        FROM kingdom_yearly
        WHERE indicator = 'Tourists Number (Overnight Visitors)'
          AND tourism_type IN ('Inbound', 'Domestic')
    )
    SELECT
        year,
        tourism_type,
        ROUND(value) AS tourists_000,
        ROUND(100.0 * (value - prev_value) / prev_value, 1) AS growth_pct
    FROM tourists
    ORDER BY tourism_type, year
""")

,year,tourism_type,tourists_000,growth_pct
0,2021,Domestic,63834.0,NaN
1,2022,Domestic,77837.0,21.9
2,2023,Domestic,81920.0,5.2
3,2024,Domestic,86157.0,5.2
4,2025,Domestic,93324.0,8.3
5,2021,Inbound,3477.0,NaN
6,2022,Inbound,16638.0,378.5
7,2023,Inbound,27424.0,64.8
8,2024,Inbound,29727.0,8.4
9,2025,Inbound,29255.0,-1.6


In [4]:
sql("""
    SELECT
        year,
        ROUND(SUM(CASE WHEN indicator = 'Tourists Number (Overnight Visitors)' THEN value END)) AS tourists_000,
        ROUND(SUM(CASE WHEN indicator = 'Tourists Spending' THEN value END)) AS spending_sar_mn,
        ROUND(SUM(CASE WHEN indicator = 'Average Spending per Trip' THEN value END)) AS spend_per_trip,
        ROUND(SUM(CASE WHEN indicator = 'Average Length of Stay' THEN value END), 1) AS avg_nights,
        ROUND(SUM(CASE WHEN indicator = 'Average Spending per Night' THEN value END)) AS spend_per_night
    FROM kingdom_yearly
    WHERE tourism_type = 'Inbound'
    GROUP BY year
    ORDER BY year
""")

,year,tourists_000,spending_sar_mn,spend_per_trip,avg_nights,spend_per_night
0,2021,3477.0,14716.0,4232.0,9.1,463.0
1,2022,16638.0,98295.0,5908.0,16.3,363.0
2,2023,27424.0,141234.0,5150.0,15.8,327.0
3,2024,29727.0,168524.0,5669.0,18.8,301.0
4,2025,29255.0,176629.0,6038.0,18.7,323.0


In [5]:
sql("""
    WITH yearly AS (
        SELECT purpose, year, SUM(tourists_000) AS tourists
        FROM inbound_purpose
        WHERE year IN (2024, 2025)
        GROUP BY purpose, year
    )
    SELECT
        purpose,
        ROUND(SUM(CASE WHEN year = 2024 THEN tourists END)) AS y2024,
        ROUND(SUM(CASE WHEN year = 2025 THEN tourists END)) AS y2025,
        ROUND(100.0 * (SUM(CASE WHEN year = 2025 THEN tourists END)
                     - SUM(CASE WHEN year = 2024 THEN tourists END))
              / SUM(CASE WHEN year = 2024 THEN tourists END), 1) AS change_pct
    FROM yearly
    GROUP BY purpose
    ORDER BY change_pct
""")

,purpose,y2024,y2025,change_pct
0,Visiting Friends & Relatives,5950.0,4371.0,-26.5
1,Other,1965.0,1478.0,-24.8
2,Leisure,7506.0,7040.0,-6.2
3,Business,1976.0,2251.0,13.9
4,Religious,12331.0,14116.0,14.5


In [6]:
sql("""
    WITH r AS (
        SELECT
            region,
            SUM(CASE WHEN year = 2023 THEN value END) AS t2023,
            SUM(CASE WHEN year = 2025 THEN value END) AS t2025
        FROM regions_yearly
        WHERE indicator = 'Tourists Number (Overnight Visitors)'
          AND tourism_type = 'Internal'
        GROUP BY region
    )
    SELECT
        region,
        ROUND(t2025) AS tourists_2025_000,
        ROUND(100.0 * t2025 / SUM(t2025) OVER (), 1) AS share_pct,
        ROUND(100.0 * (t2025 - t2023) / t2023, 1) AS growth_2023_2025_pct
    FROM r
    ORDER BY tourists_2025_000 DESC
""")

,region,tourists_2025_000,share_pct,growth_2023_2025_pct
0,Makkah,49064.0,40.0,23.8
1,Riyadh,24083.0,19.6,24.4
2,Eastern,16571.0,13.5,-13.4
3,Madinah,7255.0,5.9,14.7
4,Assir,6167.0,5.0,-13.1
5,Jazan,4288.0,3.5,46.0
6,Qassim,3565.0,2.9,-24.0
7,Tabuk,2979.0,2.4,0.8
8,Hail,2501.0,2.0,14.0
9,Jouf,2061.0,1.7,54.3


In [7]:
sql("""
    SELECT
        month_num,
        month,
        ROUND(AVG(CASE WHEN tourism_type = 'Inbound'  THEN tourists_000 END)) AS inbound_avg_000,
        ROUND(AVG(CASE WHEN tourism_type = 'Domestic' THEN tourists_000 END)) AS domestic_avg_000
    FROM monthly
    WHERE year >= 2023
    GROUP BY month_num, month
    ORDER BY month_num
""")

,month_num,month,inbound_avg_000,domestic_avg_000
0,1,January,3057.0,7484.0
1,2,February,2904.0,6846.0
2,3,March,2973.0,7532.0
3,4,April,2033.0,6882.0
4,5,May,1253.0,6104.0
5,6,June,2667.0,8983.0
6,7,July,1761.0,8292.0
7,8,August,2175.0,7206.0
8,9,September,2057.0,6165.0
9,10,October,2366.0,6452.0


In [8]:
sql("""
    SELECT
        origin_region,
        ROUND(SUM(tourists_000)) AS tourists_2025_000,
        ROUND(100.0 * SUM(tourists_000) / SUM(SUM(tourists_000)) OVER (), 1) AS share_pct
    FROM inbound_origin
    WHERE year = 2025
    GROUP BY origin_region
    ORDER BY tourists_2025_000 DESC
""")

,origin_region,tourists_2025_000,share_pct
0,Asia,9463.0,32.3
1,Middle East,7131.0,24.4
2,GCC,5681.0,19.4
3,Europe,3945.0,13.5
4,Africa,2421.0,8.3
5,Americas,570.0,1.9
6,Other,43.0,0.1


In [ ]:
sql("""
    SELECT
        t25.rank AS rank_2025,
        t25.country,
        ROUND(t25.tourists_000) AS tourists_2025_000,
        t21.rank AS rank_2021
    FROM inbound_top10_countries AS t25
    LEFT JOIN inbound_top10_countries AS t21
        ON t25.country = t21.country
       AND t21.year = 2021
    WHERE t25.year = 2025
    ORDER BY t25.rank
""")

,rank_2025,country,tourists_2025_000,rank_2021
0,1,Egypt,3174.0,3.0
1,2,Pakistan,2805.0,4.0
2,3,India,2036.0,2.0
3,4,Indonesia,2021.0,NaN
4,5,Kuwait,1837.0,1.0
5,6,Bahrain,1441.0,6.0
6,7,Turkey,1166.0,NaN
7,8,Yemen,1165.0,10.0
8,9,Iraq,1110.0,NaN
9,10,UAE,1100.0,NaN


: 